In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from warnings import filterwarnings
filterwarnings('ignore')

In [ ]:
from sklearn.preprocessing import LabelEncoder,MinMaxScaler,StandardScaler
from sklearn.model_selection import train_test_split,GridSearchCV
from imblearn.over_sampling import SMOTE

In [ ]:
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB
from mlxtend.classifier import StackingClassifier

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
df_train=pd.read_csv('C:/Users/Sushree Jena/Desktop/imarticus/machine learning/zindi/financial_inclusion/Train.csv')
df_train.shape

In [ ]:
df_train.head()

In [ ]:
df_test=pd.read_csv('C:/Users/Sushree Jena/Desktop/imarticus/machine learning/zindi/financial_inclusion/Test.csv')
df_test.shape

In [ ]:
df_test.head()

Let's merge the two datasets for the ease of doing EDA. We will separate it again at the time of model building.

In [ ]:
df_train['source']='train'
df_test['source']='test'

In [ ]:
df=pd.concat([df_train,df_test],axis=0)
df.shape

In [ ]:
df.head()

In [ ]:
#Checking for data types

In [ ]:
df.info()

We can see that we have some categoric and some numeric columns in our dataset. We can also observe that there are no missing values in the dataset.

In [ ]:
#Summary Statistics

In [ ]:
df.describe()

In [ ]:
#Plotting independent features against target column

In [ ]:
cols=['household_size','age_of_respondent']

for col in cols:
    sns.boxplot(x='bank_account',y=col,data=df)
    plt.show()

Observations:
    families with more no. of members, generally do not have a bank account.
    Families with higher age group members, generally do not have a bank account.

In [ ]:
df.columns

In [ ]:
#Visualisations

In [ ]:
sns.countplot(x='bank_account',hue='location_type',data=df)
plt.show()

We can see it clearly that there are people from Rural area who do not have a bank account as compared to Urban area.

In [ ]:
sns.countplot(x='bank_account',hue='gender_of_respondent',data=df)
plt.show()

There are less number of females as compared to males, who has a bank account.

In [ ]:
sns.countplot(x='bank_account',hue='job_type',data=df)
plt.show()

We can see clearly that there are most instances of people from Self employed sector not having a bank account, may be because they would be dealing with hard cash for daily transactions. It is followed by people from 'Informally employed' and 'Farming and Fishing', because they may not be having regular cash inflow, or not sufficient money to save for future.

In [ ]:
sns.countplot(x='bank_account',hue='education_level',data=df)
plt.show()

There are most instances of people with Primary education level, who do not have a bank account.

In [ ]:
#Feature Engineering

Let's convert the household_size and age_of_respondent to categorical column.

In [ ]:
df.age_of_respondent.min(),df.age_of_respondent.max()

In [ ]:
df['respondent_age']=pd.cut(df.age_of_respondent,bins=[10,30,60,101],labels=['young','middle','old'])

In [ ]:
df.respondent_age.value_counts()

In [ ]:
df.household_size.min(),df.household_size.max()

In [ ]:
df['family_size']=pd.cut(df.household_size,bins=[0,3,6,22],labels=['small','medium','large'])

In [ ]:
df['family_size'].value_counts()

In [ ]:
#Dropping the respondent_age and household_size column since we do not require it now.

df=df.drop(['age_of_respondent','household_size'],axis=1)

In [ ]:
df.head()

In [ ]:
sns.countplot(x='bank_account',hue='respondent_age',data=df)
plt.show()

In [ ]:
#Creating a backup file

In [ ]:
df1=df.copy(deep=True)

There are more cases of middle aged people who do not have a bank account, may be because of meeting their family requirements.

In [ ]:
df1.head()

In [ ]:
#Separating the train and test set

In [ ]:
df_train=df1.loc[df['source']=='train']
df_test=df1.loc[df['source']=='test']

In [ ]:
df_train.shape,df_test.shape

In [ ]:
df_train1=df_train.drop(['uniqueid','source'],axis=1)

In [ ]:
df_test1=df_test.drop(['uniqueid','source','bank_account'],axis=1)

In [ ]:
df_train1.shape,df_test1.shape

In [ ]:
#Dummy Encoding the Categorical Variables

In [ ]:
#for train set

In [ ]:
#Label Encoding the Target Variable

In [ ]:
le=LabelEncoder()

In [ ]:
df_train1['bank_account']=le.fit_transform(df_train1['bank_account'])

In [ ]:
df_train1['bank_account'].value_counts()

We can see that the class is not balanced. So we need to balance it using SMOTE.

In [ ]:
#One Hot Ecoding of other categorical variables

In [ ]:
df_train1_dummy=pd.get_dummies(df_train1,drop_first=True)

In [ ]:
#for test set

In [ ]:
df_test1_dummy=pd.get_dummies(df_test1,drop_first=True)

In [ ]:
#Splitting into X and y

In [ ]:
X_train=df_train1_dummy.drop('bank_account',axis=1)
y_train=df_train1_dummy['bank_account']

In [ ]:
X_test=df_test1_dummy

In [ ]:
X_train.shape,y_train.shape,X_test.shape

In [ ]:
#Balancing the target class in train data set

In [ ]:
oversample=SMOTE()

In [ ]:
X_train,y_train=oversample.fit_resample(X_train,y_train)

In [ ]:
y_train.value_counts()

We can see that the classes are balanced now.

In [ ]:
X_train.shape,y_train.shape,X_test.shape

In [ ]:
#Scaling the data

In [ ]:
sc=StandardScaler()

X_train_scaled=sc.fit_transform(X_train)
X_test_scaled=sc.transform(X_test)

In [ ]:
#Model Building

1.Logistic Regression

In [ ]:
lr=LogisticRegression()

lr.fit(X_train_scaled,y_train)

y_pred_train=lr.predict(X_train_scaled)
y_pred_test=lr.predict(X_test_scaled)

print('Training Accuracy: ',accuracy_score(y_train,y_pred_train))

submit_lr=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_lr.to_csv('financial_inclusion.submit_lr.csv',index=False)

2.DecisionTree Classifier

In [ ]:
dt=DecisionTreeClassifier()

dt.fit(X_train_scaled,y_train)

y_pred_train=dt.predict(X_train_scaled)
y_pred_test=dt.predict(X_test_scaled)

print('Training Accuracy: ',accuracy_score(y_train,y_pred_train))

submit_dt=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_dt.to_csv('financial_inclusion.submit_dt.csv',index=False)

3.RandomForest Classifier

In [ ]:
rf=RandomForestClassifier()

rf.fit(X_train_scaled,y_train)

y_pred_train=rf.predict(X_train_scaled)
y_pred_test=rf.predict(X_test_scaled)

print('Training Accuracy: ',accuracy_score(y_train,y_pred_train))

submit_rf=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_rf.to_csv('financial_inclusion.submit_rf.csv',index=False)

4.GridSearchCV_RF

In [ ]:
tuned_parameters={'n_estimators':[85,100],'min_samples_split':[15,20],'max_depth':[8,10]}

gs_model=GridSearchCV(estimator=rf,param_grid=tuned_parameters,cv=5)
gs_model.fit(X_train_scaled,y_train)

print('best parameters are: ',gs_model.best_params_)

In [ ]:
rf1=RandomForestClassifier(n_estimators=100,min_samples_split=15,max_depth=10)

rf1.fit(X_train_scaled,y_train)

y_pred_train=rf1.predict(X_train_scaled)
y_pred_test=rf1.predict(X_test_scaled)

print('Training Accuracy: ',accuracy_score(y_train,y_pred_train))

submit_rf1=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_rf1.to_csv('financial_inclusion.submit_rf1.csv',index=False)

5.GradientBoostingClassifier

In [ ]:
gb=GradientBoostingClassifier()

gb.fit(X_train_scaled,y_train)

y_pred_train=gb.predict(X_train_scaled)
y_pred_test=gb.predict(X_test_scaled)

print('Training Accuracy: ',accuracy_score(y_train,y_pred_train))

submit_gb=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_gb.to_csv('financial_inclusion.submit_gb.csv',index=False)

6.AdaBoostClassifier

In [ ]:
ab=AdaBoostClassifier()

ab.fit(X_train_scaled,y_train)

y_pred_train=ab.predict(X_train_scaled)
y_pred_test=ab.predict(X_test_scaled)

print('Training Accuracy: ',accuracy_score(y_train,y_pred_train))

submit_ab=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_ab.to_csv('financial_inclusion.submit_ab.csv',index=False)

7.XGBClassifier

In [ ]:
xgb=XGBClassifier()

xgb.fit(X_train_scaled,y_train)

y_pred_train=xgb.predict(X_train_scaled)
y_pred_test=xgb.predict(X_test_scaled)

print('Training Accuracy: ',accuracy_score(y_train,y_pred_train))

submit_xgb=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_xgb.to_csv('financial_inclusion.submit_xgb.csv',index=False)

8.LGBMClassifier

In [ ]:
lgb=LGBMClassifier()

lgb.fit(X_train_scaled,y_train)

y_pred_train=lgb.predict(X_train_scaled)
y_pred_test=lgb.predict(X_test_scaled)

print('Training Accuracy: ',accuracy_score(y_train,y_pred_train))

submit_lgb=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_lgb.to_csv('financial_inclusion.submit_lgb.csv',index=False)

9.SGDClassifier

In [ ]:
sgd=SGDClassifier()

sgd.fit(X_train_scaled,y_train)

y_pred_train=sgd.predict(X_train_scaled)
y_pred_test=sgd.predict(X_test_scaled)

print('Training Accuracy: ',accuracy_score(y_train,y_pred_train))

submit_sgd=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_sgd.to_csv('financial_inclusion.submit_sgd.csv',index=False)

10.CatBoostClassifier

In [ ]:
cb=CatBoostClassifier(logging_level='Silent')

cb.fit(X_train_scaled,y_train)

y_pred_train=cb.predict(X_train_scaled)
y_pred_test=cb.predict(X_test_scaled)

print('Training Accuracy: ',accuracy_score(y_train,y_pred_train))

submit_cb=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_cb.to_csv('financial_inclusion.submit_cb.csv',index=False)

11.Stacking

In [ ]:
dt1=DecisionTreeClassifier()

stack=StackingClassifier(classifiers=[dt,gb,lgb,xgb,cb],meta_classifier=dt1)

model_stack=stack.fit(X_train,y_train)

pred_stack_train=model_stack.predict(X_train)
pred_stack_test=model_stack.predict(X_test)

print('Training Accuracy: ',accuracy_score(y_train,pred_stack_train))

submit_stack=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": pred_stack_test})
submit_stack.to_csv('financial_inclusion.submit_stack.csv',index=False)

In [ ]:
xgb1=XGBClassifier()

stack=StackingClassifier(classifiers=[xgb,cb],meta_classifier=xgb1)

model_stack=stack.fit(X_train,y_train)

pred_stack_train=model_stack.predict(X_train)
pred_stack_test=model_stack.predict(X_test)

print('Training Accuracy: ',accuracy_score(y_train,pred_stack_train))

submit_stack1=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": pred_stack_test})
submit_stack1.to_csv('financial_inclusion.submit_stack1.csv',index=False)

12.Nueral Network

In [ ]:
classifier=Sequential()

classifier.add(Dense(34,kernel_initializer='uniform',activation='relu',input_dim=34))
classifier.add(Dense(34,kernel_initializer='uniform',activation='relu'))

classifier.add(Dense(1,kernel_initializer='uniform',activation='sigmoid'))

classifier.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

classifier.summary()

classifier.fit(X_train,y_train,batch_size=32,epochs=50)

y_pred_test=classifier.predict(X_test)

y_pred_test=np.round(y_pred_test)

y_pred_train=classifier.predict(X_train)

y_pred_train=np.round(y_pred_train)

print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

y_pred_test=y_pred_test.flatten()

y_pred_test

submit_NN=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_NN.to_csv('financial_inclusion.submit_NN.csv',index=False)

In [ ]:
classifier=Sequential()

classifier.add(Dense(34,kernel_initializer='uniform',activation='relu',input_dim=34))
classifier.add(Dense(34,kernel_initializer='uniform',activation='relu'))

classifier.add(Dense(1,kernel_initializer='uniform',activation='sigmoid'))

classifier.compile(optimizer='RMSProp',loss='binary_crossentropy',metrics=['accuracy'])

classifier.summary()

classifier.fit(X_train,y_train,batch_size=32,epochs=50)

y_pred_test=classifier.predict(X_test)

y_pred_test=np.round(y_pred_test)

y_pred_train=classifier.predict(X_train)

y_pred_train=np.round(y_pred_train)

print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

y_pred_test=y_pred_test.flatten()

y_pred_test

submit_NN1=pd.DataFrame({"uniqueid": df_test["uniqueid"] + " x " + df_test["country"],
                           "bank_account": y_pred_test})
submit_NN1.to_csv('financial_inclusion.submit_NN1.csv',index=False)